# 02 · Data Preprocessing

Runs the full preprocessing pipeline (clean -> encode -> balance -> split -> scale) on a small sample, and inspects the effect of each step.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src import config


In [ ]:
from src.data.loader import load_all_data
from src.data.preprocessing import clean_data, encode_labels, balance_classes, split_data, scale_features

raw_df = load_all_data(nrows_per_file=500, save_merged_csv=False)
print("Raw shape:", raw_df.shape)

## Step 1: Clean

In [ ]:
cleaned_df = clean_data(raw_df)
print("Cleaned shape:", cleaned_df.shape)

## Step 2: Encode labels

In [ ]:
encoded_df = encode_labels(cleaned_df)
encoded_df[["Category", "Label"]].drop_duplicates().sort_values("Label")

## Step 3: Balance classes

Before/after comparison -- this is the step that addresses CICIoT2023's severe class imbalance (ratios up to ~5,751:1 in the full dataset).

In [ ]:
print("Before balancing:")
print(encoded_df["Category"].value_counts())

balanced_df = balance_classes(encoded_df, max_per_class=300, min_per_class=50)

print("\nAfter balancing:")
print(balanced_df["Category"].value_counts())

## Step 4: Train/validation/test split

In [ ]:
train_df, val_df, test_df = split_data(balanced_df)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

## Step 5: Scale features

In [ ]:
train_df, val_df, test_df = scale_features(train_df, val_df, test_df)
train_df[config.FEATURE_COLUMNS].describe().T[["mean", "std"]]  # mean should be ~0, std ~1 after StandardScaler

## Run the full pipeline in one call

For actual project use, just call `run_preprocessing()` directly -- it performs all of the above steps and writes train.csv / validation.csv / test.csv.

In [ ]:
from src.data.preprocessing import run_preprocessing

# Uncomment to actually run and overwrite data/processed/*.csv:
# train_df, val_df, test_df = run_preprocessing(nrows_per_file=500, max_per_class=300, min_per_class=50)